In [1]:
from sympy.parsing.sympy_parser import parse_expr
from sympy import Symbol

# Define symbols if needed, or let parse_expr create them automatically
x = Symbol('x')
y = Symbol('y')

expression_string = "x**2 + 2*y + sin(pi/4)"
parsed_expression = parse_expr(expression_string)

print(parsed_expression)
# Output: 2*y + x**2 + sin(pi/4) (The terms might be reordered for canonical representation)
print(type(parsed_expression))
# Output: <class 'sympy.core.add.Add'>


x**2 + 2*y + sqrt(2)/2
<class 'sympy.core.add.Add'>


In [10]:
from sympy import symbols, srepr, sin
x = symbols('x')
expr = sin(x) + 1

# Method 1: srepr (String Representation)
print(srepr(expr))
# Output: Add(1, sin(Symbol('x')))

# Method 2: Detailed Tree Print (Requires 1.14+)
from sympy.printing import pprint
from sympy.printing.tree import print_tree
pprint(expr) #, tree=True)
print_tree(expr, assumptions=False)


Add(sin(Symbol('x')), Integer(1))
sin(x) + 1
Add: sin(x) + 1
+-One: 1
+-sin: sin(x)
  +-Symbol: x



In [4]:
from sympy import symbols, srepr, sin

x, y = symbols('x y')
expr = sin(x*y)/2 - x**2 + 1/y

# Get the string representation of the expression tree
tree_repr = srepr(expr)

print(tree_repr)

Add(Mul(Integer(-1), Pow(Symbol('x'), Integer(2))), Mul(Rational(1, 2), sin(Mul(Symbol('x'), Symbol('y')))), Pow(Symbol('y'), Integer(-1)))


In [13]:
from sympy.parsing.sympy_parser import (
    parse_expr,
    standard_transformations,
    implicit_multiplication_application,
)
from sympy import symbols

x, y = symbols('x y')

# Combine standard transformations with the implicit multiplication one
transformations = (standard_transformations + (implicit_multiplication_application,))

expression_string = "2x + y"

# Parse the expression
parsed_expression = parse_expr(expression_string, transformations=transformations)

print(parsed_expression)
print(type(parsed_expression))


2*x + y
<class 'sympy.core.add.Add'>


In [15]:
expr = parse_expr("A x**2 + B x y + C y**2 + D x + E y + F", transformations=transformations)

In [17]:
print_tree(expr, assumptions=False)

Add: A*x**2 + B*x*y + C*y**2 + D*x + F + E*y
+-Symbol: F
+-Mul: E*y
| +-Exp1: E
| +-Symbol: y
+-Mul: A*x**2
| +-Symbol: A
| +-Pow: x**2
|   +-Symbol: x
|   +-Integer: 2
+-Mul: C*y**2
| +-Symbol: C
| +-Pow: y**2
|   +-Symbol: y
|   +-Integer: 2
+-Mul: D*x
| +-Symbol: D
| +-Symbol: x
+-Mul: B*x*y
  +-Symbol: B
  +-Symbol: x
  +-Symbol: y



In [18]:
from sympy import symbols, Eq
x, y = symbols('x y')
eq = Eq(x + 2, y)
print(eq.lhs)  # Output: x + 2
print(eq.rhs)  # Output: y

x + 2
y


In [20]:
eq = Eq(expr, 0)

In [21]:
eq

Eq(A*x**2 + B*x*y + C*y**2 + D*x + F + E*y, 0)

In [22]:
print_tree(eq, assumptions=False)

Equality: Eq(A*x**2 + B*x*y + C*y**2 + D*x + F + E*y, 0)
+-Add: A*x**2 + B*x*y + C*y**2 + D*x + F + E*y
| +-Symbol: F
| +-Mul: E*y
| | +-Exp1: E
| | +-Symbol: y
| +-Mul: A*x**2
| | +-Symbol: A
| | +-Pow: x**2
| |   +-Symbol: x
| |   +-Integer: 2
| +-Mul: C*y**2
| | +-Symbol: C
| | +-Pow: y**2
| |   +-Symbol: y
| |   +-Integer: 2
| +-Mul: D*x
| | +-Symbol: D
| | +-Symbol: x
| +-Mul: B*x*y
|   +-Symbol: B
|   +-Symbol: x
|   +-Symbol: y
+-Zero: 0



In [23]:
from sympy import symbols, Matrix, expand

# Define symbols
x, y = symbols('x y')
# Example quadratic equation: 3x^2 + 4xy + 5y^2 + 6x + 7y + 8 = 0
# A=3, B=4, C=5, D=6, E=7, F=8

# Define vector v
v = Matrix([[x], [y]])

# Define symmetric Matrix M (for quadratic terms)
# M = [[A, B/2], [B/2, C]]
M = Matrix([[3, 4/2], [4/2, 5]])

# Define Linear Matrix L (for linear terms)
# L = [D, E]
L = Matrix([[6, 7]])

# Constant term
F = 8

# Formulate quadratic expression: v.T * M * v + L * v + F
matrix_expr = (v.transpose() * M * v)[0,0] + (L * v)[0,0] + F

# Verify by expanding the matrix expression
print("Matrix Expression:", (v.transpose() * M * v)[0,0] + (L * v)[0,0] + F)
print("Expanded:", expand(matrix_expr))


Matrix Expression: x*(3*x + 2.0*y) + 6*x + y*(2.0*x + 5*y) + 7*y + 8
Expanded: 3*x**2 + 4.0*x*y + 6*x + 5*y**2 + 7*y + 8


In [24]:
import sympy as sp

# 1. Define symbols
x, y = sp.symbols('x y')

# 2. Define the quadratic expression
# Example: 3x^2 + 4xy + 5y^2 + 2x + 6y + 10
expr = 3*x**2 + 4*x*y + 5*y**2 + 2*x + 6*y + 10

# 3. Extract coefficients using Poly
# poly() sorts the terms automatically
poly_expr = sp.Poly(expr, x, y)
coeffs = poly_expr.coeffs()
monoms = poly_expr.monoms()

# 4. Construct the symmetric Matrix A (for x^2, xy, y^2)
# The coefficient of xy is split between A[0,1] and A[1,0]
a = poly_expr.coeff_monomial(x**2)
b = poly_expr.coeff_monomial(x*y) / 2
c = poly_expr.coeff_monomial(y**2)
A = sp.Matrix([[a, b], [b, c]])

# 5. Construct the Linear Vector B (for x, y)
d = poly_expr.coeff_monomial(x)
e = poly_expr.coeff_monomial(y)
B = sp.Matrix([d, e])

# 6. Constant
f = poly_expr.coeff_monomial(1)

# 7. Define Variable Vector v
v = sp.Matrix([x, y])

# 8. Assemble Matrix Expression: vT * A * v + B.T * v + f
matrix_form = (v.T * A * v)[0] + (B.T * v)[0] + f

# Verify
print("Quadratic Matrix A:")
sp.pprint(A)
print("\nLinear Vector B:")
sp.pprint(B)
print("\nOriginal Expression:", expr)
print("Matrix Expression:", sp.expand(matrix_form))
print("Match:", sp.simplify(matrix_form - expr) == 0)


Quadratic Matrix A:
⎡3  2⎤
⎢    ⎥
⎣2  5⎦

Linear Vector B:
⎡2⎤
⎢ ⎥
⎣6⎦

Original Expression: 3*x**2 + 4*x*y + 2*x + 5*y**2 + 6*y + 10
Matrix Expression: 3*x**2 + 4*x*y + 2*x + 5*y**2 + 6*y + 10
Match: True


In [25]:
expr

3*x**2 + 4*x*y + 2*x + 5*y**2 + 6*y + 10

In [26]:
str(expr)

'3*x**2 + 4*x*y + 2*x + 5*y**2 + 6*y + 10'

In [27]:
from sympy import symbols, Pow
from sympy.printing.str import StrPrinter

class CustomStrPrinter(StrPrinter):
    """A custom printer to use ^ for exponentiation."""
    def _print_Pow(self, expr, **kwargs):
        # Format as base^exp
        return f"{self._print(expr.base, **kwargs)}^{self._print(expr.exp, **kwargs)}"

def sympy_to_str_with_caret(expr):
    """Converts a SymPy expression to a string using ^ for powers."""
    printer = CustomStrPrinter()
    return printer.doprint(expr)

# --- Example Usage ---
if __name__ == '__main__':
    x, y, z = symbols('x y z')
    expr = x**2 + 2*x*y + y**3
    
    # Standard SymPy string output uses **
    standard_str = str(expr)
    print(f"Standard string: {standard_str}")
    
    # Custom string output uses ^
    custom_str = sympy_to_str_with_caret(expr)
    print(f"Custom string:   {custom_str}")

    expr_complex = (x**2 + 1/y)**z
    custom_str_complex = sympy_to_str_with_caret(expr_complex)
    print(f"Custom complex:  {custom_str_complex}")


Standard string: x**2 + 2*x*y + y**3
Custom string:   x^2 + 2*x*y + y^3
Custom complex:  x^2 + y^-1^z


In [22]:
from sympy.parsing.sympy_parser import (
    parse_expr, 
    convert_xor,
    standard_transformations,
    implicit_multiplication_application,
)

transformations = (standard_transformations + (implicit_multiplication_application,convert_xor,))
E_symbol = symbols('E')
eq_str = "A x^2 + B x y + C y^2 + D x + E y + F = 0"
lhs_str, rhs_str = eq_str.split('=')
expr = parse_expr(lhs_str, local_dict={'E': E_symbol}, transformations=transformations)
eq = Eq(expr, 0)
eq

NameError: name 'symbols' is not defined

In [75]:
A=2
B=3
C=4
D=5
E=7
F=-8

In [76]:
B**2 - 4 * A * C

-23

In [77]:
2*C*D - B*E, 2*A*E - B*D

(19, 13)

In [78]:
import ggblab

In [83]:
%%ggblab
A=2
B=3
C=4
D=5
E=6
F=0

['A', 'B', 'C', 'D', 'E', 'F']

In [84]:
%ggblab eq1: {eq_str}

['eq1']

In [95]:
import sympy
discr = sympy.discriminant(expr, (x, y))
discr

-4*A*F + D**2 + y**2*(-4*A*C + B**2) + y*(-4*A*E + 2*B*D)

In [98]:
discr.coeff(y, 2)

-4*A*C + B**2

In [90]:
y

y

In [103]:
r2 = await ggb.function('getXML')

In [102]:
print(r1)

<?xml version="1.0" encoding="utf-8"?>
<geogebra format="5.0" version="5.2.909.9" app="suite" subApp="graphing" platform="w" id="9a879c40-effb-4ec3-b764-ae427c38fa2f" xsi:noNamespaceSchemaLocation="https://www.geogebra.org/apps/xsd/ggb.xsd" xmlns="" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">
<gui>
	<window width="640" height="1071"/>
	<perspectives>
<perspective id="tmp">
	<panes>
	<pane location="" divider="0.738562091503268" orientation="0"/>
</panes>
	<views>
	<view id="4097" visible="false" inframe="false" stylebar="true" location="1,1,1,1" size="400" window="100,100,700,550"/>
	<view id="512" toolbar="0 | 1 501 5 19 , 67 | 2 15 45 18 , 7 37 | 514 3 9 , 13 44 , 47 | 16 51 | 551 550 11 ,  20 22 21 23 , 55 56 57 , 12 | 69 | 510 511 , 512 513 | 533 531 , 534 532 , 522 523 , 537 536 , 535 , 538 | 521 520 | 36 , 38 49 560 | 571 30 29 570 31 33 | 17 | 540 40 41 42 , 27 28 35 , 6 , 502" visible="false" inframe="false" stylebar="false" location="1,1,1" size="500" window="100,10

In [104]:
print(r2)

<?xml version="1.0" encoding="utf-8"?>
<geogebra format="5.0" version="5.2.909.9" app="suite" subApp="3d" platform="w" id="9a879c40-effb-4ec3-b764-ae427c38fa2f" xsi:noNamespaceSchemaLocation="https://www.geogebra.org/apps/xsd/ggb.xsd" xmlns="" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">
<gui>
	<window width="640" height="1071"/>
	<perspectives>
<perspective id="tmp">
	<panes>
	<pane location="" divider="0.7105508870214753" orientation="0"/>
</panes>
	<views>
	<view id="4097" visible="false" inframe="false" stylebar="true" location="1,1,1,1" size="400" window="100,100,700,550"/>
	<view id="8" toolbar="1001 | 1002 | 1003  || 1005 | 1004 || 1006 | 1007 | 1010 || 1008 | 1009 || 6" visible="false" inframe="false" stylebar="false" location="1,3,3" size="300" window="100,100,600,400"/>
	<view id="1" visible="false" inframe="false" stylebar="false" location="1,3" size="500" window="100,100,600,400"/>
	<view id="4" toolbar="0 || 2020 , 2021 , 2022 || 2001 , 2003 , 2002 , 2004 , 2005 

In [106]:
from xmldiff import main

In [107]:
main.diff_texts(r1, r2)

ValueError: Unicode strings with encoding declaration are not supported. Please use bytes input or XML fragments without declaration.

In [111]:
import difflib

In [112]:
diff = difflib.unified_diff(
    r.splitlines(),
    r2.splitlines(),
    lineterm=''
)


In [114]:
list(diff)

['--- ',
 '+++ ',
 '@@ -1,7 +1,7 @@',
 ' <?xml version="1.0" encoding="utf-8"?>',
 '-<geogebra format="5.0" version="5.2.909.9" app="suite" subApp="graphing" platform="w" id="9a879c40-effb-4ec3-b764-ae427c38fa2f" xsi:noNamespaceSchemaLocation="https://www.geogebra.org/apps/xsd/ggb.xsd" xmlns="" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">',
 '+<geogebra format="5.0" version="5.2.909.9" app="suite" subApp="3d" platform="w" id="9a879c40-effb-4ec3-b764-ae427c38fa2f" xsi:noNamespaceSchemaLocation="https://www.geogebra.org/apps/xsd/ggb.xsd" xmlns="" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">',
 ' <gui>',
 '-\t<window width="971" height="1071"/>',
 '+\t<window width="640" height="1071"/>',
 ' \t<perspectives>',
 ' <perspective id="tmp">',
 ' \t<panes>',
 '@@ -9,30 +9,30 @@',
 ' </panes>',
 ' \t<views>',
 ' \t<view id="4097" visible="false" inframe="false" stylebar="true" location="1,1,1,1" size="400" window="100,100,700,550"/>',
 '-\t<view id="512" toolbar="0 | 1 501 5 

In [115]:
print(r1)

<?xml version="1.0" encoding="utf-8"?>
<geogebra format="5.0" version="5.2.909.9" app="suite" subApp="graphing" platform="w" id="9a879c40-effb-4ec3-b764-ae427c38fa2f" xsi:noNamespaceSchemaLocation="https://www.geogebra.org/apps/xsd/ggb.xsd" xmlns="" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">
<gui>
	<window width="640" height="1071"/>
	<perspectives>
<perspective id="tmp">
	<panes>
	<pane location="" divider="0.738562091503268" orientation="0"/>
</panes>
	<views>
	<view id="4097" visible="false" inframe="false" stylebar="true" location="1,1,1,1" size="400" window="100,100,700,550"/>
	<view id="512" toolbar="0 | 1 501 5 19 , 67 | 2 15 45 18 , 7 37 | 514 3 9 , 13 44 , 47 | 16 51 | 551 550 11 ,  20 22 21 23 , 55 56 57 , 12 | 69 | 510 511 , 512 513 | 533 531 , 534 532 , 522 523 , 537 536 , 535 , 538 | 521 520 | 36 , 38 49 560 | 571 30 29 570 31 33 | 17 | 540 40 41 42 , 27 28 35 , 6 , 502" visible="false" inframe="false" stylebar="false" location="1,1,1" size="500" window="100,10

In [116]:
import xml.etree.ElementTree as ET

In [149]:
attrib = ET.fromstring(r2).attrib
attrib['app'], attrib['subApp']

('suite', '3d')

In [147]:
ET.fromstring(r1).attrib

{'format': '5.0',
 'version': '5.2.909.9',
 'app': 'suite',
 'subApp': 'graphing',
 'platform': 'w',
 'id': '9a879c40-effb-4ec3-b764-ae427c38fa2f',
 '{http://www.w3.org/2001/XMLSchema-instance}noNamespaceSchemaLocation': 'https://www.geogebra.org/apps/xsd/ggb.xsd'}

In [145]:
root.attrib

{'format': '5.0',
 'version': '5.2.909.9',
 'app': 'suite',
 'subApp': 'graphing',
 'platform': 'w',
 'id': '9a879c40-effb-4ec3-b764-ae427c38fa2f',
 '{http://www.w3.org/2001/XMLSchema-instance}noNamespaceSchemaLocation': 'https://www.geogebra.org/apps/xsd/ggb.xsd'}

In [131]:
root.text

'\n'

In [132]:
for elem in root.iter():
    print(elem)

<Element 'geogebra' at 0x3205910d0>
<Element 'gui' at 0x320591120>
<Element 'window' at 0x320591170>
<Element 'perspectives' at 0x3205911c0>
<Element 'perspective' at 0x320591210>
<Element 'panes' at 0x320591260>
<Element 'pane' at 0x3205912b0>
<Element 'views' at 0x320591300>
<Element 'view' at 0x320591350>
<Element 'view' at 0x3205913a0>
<Element 'view' at 0x3205913f0>
<Element 'view' at 0x320591440>
<Element 'view' at 0x320591490>
<Element 'view' at 0x3205914e0>
<Element 'view' at 0x320591530>
<Element 'view' at 0x320591580>
<Element 'view' at 0x3205915d0>
<Element 'view' at 0x320591620>
<Element 'view' at 0x3205916c0>
<Element 'toolbar' at 0x320591710>
<Element 'input' at 0x320591760>
<Element 'dockBar' at 0x3205917b0>
<Element 'labelingStyle' at 0x320591800>
<Element 'font' at 0x320591850>
<Element 'euclidianView' at 0x3205918a0>
<Element 'viewNumber' at 0x3205918f0>
<Element 'size' at 0x320591940>
<Element 'coordSystem' at 0x320591990>
<Element 'evSettings' at 0x3205919e0>
<Eleme

In [159]:
r = await ggb.function('getValuie', ['f'])

In [2]:
import ggblab

In [3]:
%ggb api getVersion()

[ggblab] created GeoGebra singleton and stored as 'ggb' in user namespace


'5.2.909.9'

In [7]:
await ggb.function('getValueString', ['f'])

'f: -4.72x + 4.2y = 6.7692'

In [8]:
await ggb.function('getValueString', ['g'])

'g: -4.72x + 4.2y = 6.7692'

In [10]:
await ggb.function('getValueString', ['h'])

'h = 6.3181009805162'

In [11]:
await ggb.function('getValueString', ['f'])

'f: X = (0, 0, 2) + λ (-1, 0, -2)'

In [14]:
await ggb.function('getObjectType', ['f'])

'ray'

In [122]:
r1 = await ggb.function('getValueString', ['f'])
r1

'f: y = -1.4x + 8.7'

In [123]:
r2 = await ggb.function('getValueString', ['c'])
r2

'c: x² + y² = 16'

In [124]:
x, y = symbols('x y')#, real=False)

In [125]:
from sympy.parsing.sympy_parser import (
    parse_expr, 
    convert_xor,
    standard_transformations,
    implicit_multiplication_application,
)
from sympy import symbols, Eq

transformations = (standard_transformations + (implicit_multiplication_application,convert_xor,))


_, eq_str = r1.split(':')

lhs_str, rhs_str = eq_str.split('=')
expr_lhs = parse_expr(lhs_str.replace("²", "^2"), transformations=transformations)
expr_rhs = parse_expr(rhs_str.replace("²", "^2"), transformations=transformations)
eq1 = Eq(expr_lhs, expr_rhs)
eq1

_, eq_str = r2.split(':')

lhs_str, rhs_str = eq_str.split('=')
expr_lhs = parse_expr(lhs_str.replace("²", "^2"), transformations=transformations)
expr_rhs = parse_expr(rhs_str.replace("²", "^2"), transformations=transformations)
eq2 = Eq(expr_lhs, expr_rhs)
eq2

Eq(x**2 + y**2, 16)

In [126]:
eq1

Eq(y, 8.7 - 1.4*x)

In [127]:
from sympy import symbols, Eq, solve, nonlinsolve, linsolve

nonlinsolve([eq1, eq2], [x, y])
# linsolve([eq1, eq2], [x, y])

{(4.11486486486486 - 1.79817343002314*I, 2.93918918918919 + 2.5174428020324*I), (4.11486486486486 + 1.79817343002314*I, 2.93918918918919 - 2.5174428020324*I)}

## 二元二次方程式

In [2]:
import ggblab

In [3]:
%ggb api newConstruction()

[ggblab] created GeoGebra singleton and stored as 'ggb' in user namespace


In [4]:
%%ggblab
a = Slider(-5, 5)
b = Slider(-5, 5)
c = Slider(-5, 5)
d = Slider(-5, 5)
e = Slider(-5, 5)
f = Slider(-5, 5)

['a', 'b', 'c', 'd', 'e', 'f']

In [5]:
%ggb y = a x + b

['g']

In [7]:
await ggb.function("getValueString", ['g'])

'g(x) = 1.7 x + 1.2'

In [8]:
%ggblab Curve(t, a t + b, t, -5, 5) 

['h']

In [9]:
await ggb.function("getValueString", ['h'])

'h:(t, 1.7 t + 1.2)'

In [10]:
%ggb eq1: a * c x - c y + b * c = 0

['eq1']

In [11]:
await ggb.function("getValueString", ['eq1'])

'eq1: 1.7 * 1 x - 1 y + 1.2 * 1 = 0'

In [12]:
%ggb Line((0, b), Vector((1, a)))

['i']

In [20]:
%ggb Line((0,b), (1,a+b))

['k']

In [13]:
await ggb.function("getValueString", ['i'])

'i: -1.7x + y = 1.2'

In [14]:
%ggb j: a x + b

['j']

In [15]:
await ggb.function("getValueString", ['j'])

'j(x) = 1.7 x + 1.2'

In [23]:
%ggb (0,b)

['A']

In [24]:
%ggb (1, a+b)

['B']

In [16]:
from ggblab_extra import ConstructionIO
from ggblab_extra import ConstructionTreeParser

In [27]:
df = await ConstructionIO.initialize_dataframe(ggb, use_applet=True)

In [29]:
df["Name", "Type", "Command", "Value"]

Name,Type,Command,Value
str,str,str,str
"""a""","""numeric""",null,"""a = 0.5"""
"""b""","""numeric""",null,"""b = 2"""
"""c""","""numeric""",null,"""c = 1"""
"""d""","""numeric""",null,"""d = 0"""
"""e""","""numeric""",null,"""e = 0"""
"""f""","""numeric""",null,"""f = 0"""
"""g""","""function""","""a x + b""","""g(x) = 0.5 x + 2"""
"""h""","""curvecartesian""","""Curve(t, a t + b, t, -5, 5)""","""h:(t, 0.5 t + 2)"""
"""eq1""","""line""","""a c x - c y + b c = 0""","""eq1: 0.5 * 1 x - 1 y + 2 * 1 =…"


In [109]:
%ggb j(3)

['k']

In [110]:
%ggb h(3)

['A']

In [25]:
%%ggb
d * x^2 + e x y + f y^2 + (a c) x - c y + (b c) = 0

['eq2']

In [26]:
await ggb.function("getValueString", ['eq2'])

'eq2: 0 x² + 0 x y + 0 y² + 0.5 * 1 x - 1 y + 2 * 1 = 0'

In [165]:
await ggb.command('Coefficients(eq2)')

'l1'

In [166]:
%ggb api getValueString(l1)

'l1 = {0, 0, 0.6, 0, 1.26, -0.6}'

In [167]:
%ggb {{1,2,3},{4,5,6},{7,8,9}}

['m1']

In [170]:
%ggb api getValueString(m1)

'm1 = {{1, 2, 3}, {4, 5, 6}, {7, 8, 9}}'

In [171]:
from sympy import symbols, Matrix, expand

In [173]:
r = await ggb.function("getValueString", ['m1'])

In [179]:
m = Matrix(ggb.parser.tokenize(r)[2])

In [186]:
m.row(0)

Matrix([[1, 2, 3]])

In [187]:
m.col(0)

Matrix([
[1],
[4],
[7]])

In [188]:
from sympy import symbols, Matrix, cos, sin, pi

# Define symbolic variables for transformation parameters
a, b, c, d, tx, ty = symbols('a b c d tx ty')

# Define the 2D affine transformation matrix
affine_matrix_2d = Matrix([
    [a, b, tx],
    [c, d, ty],
    [0, 0, 1]
])

print("2D Affine Matrix:")
print(affine_matrix_2d)

# Example of a 2D rotation and translation
theta = symbols('theta')
rotation_angle = pi/4 # 45 degrees

# Parameters for rotation by theta and translation by (5, -2)
a_val = cos(theta)
b_val = -sin(theta)
c_val = sin(theta)
d_val = cos(theta)
tx_val = 5
ty_val = -2

# Construct the specific transformation matrix
specific_affine_matrix = Matrix([
    [a_val, b_val, tx_val],
    [c_val, d_val, ty_val],
    [0, 0, 1]
])

print("\nSpecific Affine Matrix (Rotation and Translation):")
print(specific_affine_matrix)

# Apply the transformation to a point (x_p, y_p)
x_p, y_p = symbols('x_p y_p')
original_point = Matrix([
    [x_p],
    [y_p],
    [1]
])

transformed_point = specific_affine_matrix * original_point

print("\nTransformed Point:")
print(transformed_point)

# Evaluate for specific angle
transformed_point_eval = transformed_point.subs({theta: rotation_angle})
print(f"\nTransformed Point at theta = pi/4:")
print(transformed_point_eval.evalf())


2D Affine Matrix:
Matrix([[a, b, tx], [c, d, ty], [0, 0, 1]])

Specific Affine Matrix (Rotation and Translation):
Matrix([[cos(theta), -sin(theta), 5], [sin(theta), cos(theta), -2], [0, 0, 1]])

Transformed Point:
Matrix([[x_p*cos(theta) - y_p*sin(theta) + 5], [x_p*sin(theta) + y_p*cos(theta) - 2], [1]])

Transformed Point at theta = pi/4:
Matrix([[0.707106781186548*x_p - 0.707106781186548*y_p + 5.0], [0.707106781186548*x_p + 0.707106781186548*y_p - 2.0], [1.00000000000000]])


In [193]:
affine_matrix_2d.subs({a: cos(theta), b: -sin(theta), c: sin(theta), d: cos(theta)})

Matrix([
[cos(theta), -sin(theta), tx],
[sin(theta),  cos(theta), ty],
[         0,           0,  1]])